# SFT Fine-Tuning Pipeline — 5-Trial Automated Loop
**Track 1, Option A: SFT → DPO**

Platform: Kaggle T4 GPU (16 GB VRAM)
Base Model: Qwen/Qwen2.5-0.5B-Instruct
Dataset: HuggingFaceH4/no_robots


In [ ]:
# Cell 1 — Install Dependencies
# !pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
# !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
# !pip install --no-deps unsloth
# !pip install -U bitsandbytes
# !pip install bert_score evaluate nltk pandas tabulate
# print("Dependencies installed.")


In [ ]:
# Cell 2 — Imports
import os, gc, torch
import pandas as pd
import nltk
from datasets import load_dataset
from transformers import TrainingArguments
from evaluate import load as load_metric
from bert_score import score as bert_score_fn
from unsloth import FastLanguageModel
from trl import SFTTrainer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


In [ ]:
# Cell 3 — Configuration Constants
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN = 512
DTYPE = None  # auto-detect (float16 on T4)
LOAD_IN_4BIT = True
OUTPUT_ROOT = "/kaggle/working"


In [ ]:
# Cell 4 — 10 Manual Test Prompts + Gold References
EVAL_PROMPTS = [
    "How do I start a small business?",
    "Tell me a joke about programmers.",
    "How can I improve my communication skills?",
    "What is the capital of France?",
    "Explain machine learning like I'm 5.",
    "Give me a fun fact about space.",
    "How do I bake a chocolate cake?",
    "What's the best way to stay motivated?",
    "What are some good daily habits to build?",
    "Tell me about the importance of sleep.",
]

GOLD_REFERENCES = [
    "Start by identifying your niche, writing a business plan, registering your company, and securing initial funding.",
    "Why do programmers prefer dark mode? Because light attracts bugs!",
    "Practice active listening, maintain eye contact, use open body language, and be clear and concise in your speech.",
    "The capital of France is Paris, which is also the country's largest city.",
    "Machine learning is like teaching a computer to learn from examples, just like how you learn to recognize animals from picture books.",
    "A day on Venus is longer than a year on Venus — it takes 243 Earth days to rotate once but only 225 days to orbit the Sun!",
    "Mix flour, sugar, cocoa powder, eggs, butter, and milk together, then bake at 350°F (175°C) for about 30 minutes.",
    "Set small achievable goals, celebrate progress, maintain a routine, and remind yourself of your purpose when motivation dips.",
    "Wake up early, exercise regularly, read for at least 20 minutes, practice gratitude, and plan your day the night before.",
    "Sleep is essential for brain recovery, memory consolidation, immune function, and emotional regulation.",
]


In [ ]:
# Cell 5 — Evaluation Function (BLEU + BERTScore)
bleu_metric = load_metric("bleu")

def evaluate_model(model, tokenizer, prompts, references, trial_name=""):
    """Generate responses for prompts, compute BLEU and BERTScore F1."""
    FastLanguageModel.for_inference(model)
    model.eval()
    generated = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150,
                                 temperature=0.7, top_p=0.9,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        generated.append(response)

    # BLEU
    bleu_result = bleu_metric.compute(
        predictions=generated,
        references=[[r] for r in references]
    )
    bleu_score = bleu_result["bleu"]

    # BERTScore
    P, R, F1 = bert_score_fn(generated, references, lang="en", verbose=False)
    bert_f1 = F1.mean().item()

    # Print sample outputs
    print(f"\n{'='*60}")
    print(f"  Evaluation Results — {trial_name}")
    print(f"{'='*60}")
    for i, (p, g, r) in enumerate(zip(prompts, generated, references)):
        print(f"\n[Prompt {i+1}]: {p}")
        print(f"  Model:     {g[:200]}")
        print(f"  Reference: {r[:200]}")
    print(f"\n>>> BLEU: {bleu_score:.4f} | BERTScore F1: {bert_f1:.4f}")
    return bleu_score, bert_f1, generated


In [ ]:
# Cell 6 — Load Dataset (HuggingFaceH4/no_robots)
dataset = load_dataset("HuggingFaceH4/no_robots")

def format_to_chat(example):
    """Convert no_robots messages format to a single text string."""
    msgs = example["messages"]
    text = ""
    for m in msgs:
        role = m["role"]
        content = m["content"]
        if role == "system":
            text += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == "user":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
    return {"text": text}

train_dataset = dataset["train"].map(format_to_chat)
test_dataset = dataset["test"].map(format_to_chat)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Cell 7 — Baseline Evaluation (Before Any Fine-Tuning)
print("\n" + "="*60)
print("  BASELINE MODEL EVALUATION (Before SFT)")
print("="*60)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_bleu, base_bert, _ = evaluate_model(model, tokenizer, EVAL_PROMPTS, GOLD_REFERENCES, "Baseline")

# Clean up base model to free VRAM before trials
del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Cell 8 — Define 5 SFT Trial Configurations
TRIAL_CONFIGS = [
    {
        "trial_id": 1,
        "lora_r": 8,
        "lora_alpha": 16,
        "target_modules": ["q_proj", "v_proj"],
        "lr": 2e-4,
        "batch_size": 8,
        "epochs": 1,
        "grad_accum": 4,
    },
    {
        "trial_id": 2,
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 1e-4,
        "batch_size": 4,
        "epochs": 1,
        "grad_accum": 8,
    },
    {
        "trial_id": 3,
        "lora_r": 32,
        "lora_alpha": 64,
        "target_modules": ["q_proj", "v_proj"],
        "lr": 5e-5,
        "batch_size": 8,
        "epochs": 2,
        "grad_accum": 4,
    },
    {
        "trial_id": 4,
        "lora_r": 16,
        "lora_alpha": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 3e-4,
        "batch_size": 16,
        "epochs": 1,
        "grad_accum": 2,
    },
    {
        "trial_id": 5,
        "lora_r": 8,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lr": 1e-4,
        "batch_size": 4,
        "epochs": 2,
        "grad_accum": 4,
    },
]


In [ ]:
# Cell 9 — Automated 5-Trial Training Loop
results = []

for cfg in TRIAL_CONFIGS:
    trial_id = cfg["trial_id"]
    trial_name = f"SFT Trial {trial_id}"
    output_dir = f"{OUTPUT_ROOT}/sft_trial_{trial_id}"
    print(f"\n{'#'*60}")
    print(f"  STARTING {trial_name}")
    print(f"  Config: r={cfg['lora_r']}, modules={cfg['target_modules']}, lr={cfg['lr']}, bs={cfg['batch_size']}")
    print(f"{'#'*60}")

    # --- Load fresh base model ---
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- Apply LoRA ---
    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        target_modules=cfg["target_modules"],
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    # --- Training Arguments ---
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["grad_accum"],
        num_train_epochs=cfg["epochs"],
        learning_rate=cfg["lr"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="no",
        warmup_ratio=0.05,
        weight_decay=0.01,
        report_to="none",
    )

    # --- Trainer ---
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
    )

    # --- Train ---
    train_result = trainer.train()
    train_loss = train_result.training_loss

    # --- Get Validation Loss ---
    eval_result = trainer.evaluate()
    val_loss = eval_result.get("eval_loss", float("nan"))

    # --- Evaluate with BLEU + BERTScore ---
    bleu, bert_f1, gen_responses = evaluate_model(
        model, tokenizer, EVAL_PROMPTS, GOLD_REFERENCES, trial_name
    )

    # --- Save adapter ---
    adapter_path = f"{OUTPUT_ROOT}/sft_adapter_trial_{trial_id}"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to {adapter_path}")

    # --- Record results ---
    results.append({
        "Trial": trial_id,
        "LoRA_r": cfg["lora_r"],
        "LoRA_alpha": cfg["lora_alpha"],
        "Target_Modules": str(cfg["target_modules"]),
        "LR": cfg["lr"],
        "Batch_Size": cfg["batch_size"],
        "Epochs": cfg["epochs"],
        "Train_Loss": round(train_loss, 4),
        "Val_Loss": round(val_loss, 4),
        "BLEU": round(bleu, 4),
        "BERTScore_F1": round(bert_f1, 4),
    })

    # --- Cleanup GPU ---
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cache cleared after {trial_name}.")


In [ ]:
# Cell 10 — Results Summary & Best Model Selection
df = pd.DataFrame(results)

# Add baseline row
baseline_row = pd.DataFrame([{
    "Trial": 0, "LoRA_r": "-", "LoRA_alpha": "-", "Target_Modules": "-",
    "LR": "-", "Batch_Size": "-", "Epochs": "-",
    "Train_Loss": "-", "Val_Loss": "-",
    "BLEU": round(base_bleu, 4), "BERTScore_F1": round(base_bert, 4),
}])
df_full = pd.concat([baseline_row, df], ignore_index=True)

print("\n" + "="*60)
print("  SFT TRIAL RESULTS SUMMARY")
print("="*60)
print(df_full.to_string(index=False))

# Select best trial (highest composite of BLEU + BERTScore)
df["composite"] = df["BLEU"] + df["BERTScore_F1"]
best_idx = df["composite"].idxmax()
best_trial = df.loc[best_idx, "Trial"]
print(f"\n>>> Best SFT Trial: Trial {best_trial}")
print(f"    BLEU={df.loc[best_idx, 'BLEU']}, BERTScore={df.loc[best_idx, 'BERTScore_F1']}")

# Save CSV
csv_path = f"{OUTPUT_ROOT}/sft_trial_results.csv"
df_full.to_csv(csv_path, index=False)
print(f"Results saved to {csv_path}")


In [ ]:
# Cell 11 — Copy Best Adapter to Canonical Path
import shutil

best_src = f"{OUTPUT_ROOT}/sft_adapter_trial_{best_trial}"
best_dst = f"{OUTPUT_ROOT}/sft_best_adapter"
if os.path.exists(best_dst):
    shutil.rmtree(best_dst)
shutil.copytree(best_src, best_dst)
print(f"\nBest SFT adapter copied to: {best_dst}")
print("This adapter will be used as the base for DPO training.")
